# Homework: Image Classification — ResNet-18 Parameter Analysis & Replication

**Author:** Shuja
**Model chosen:** ResNet-18 (He et al., 2015)
**Dataset:** CIFAR-10
**Paper:** *Deep Residual Learning for Image Recognition*, He, Zhang, Ren, Sun — CVPR 2016. arXiv:1512.03385

---

## Table of contents
- [Part 1 — Paper Review & Parameter Calculation](#part1)
  - Architecture overview
  - Per-layer formulas
  - Full breakdown table
  - Total count & comparison with torchvision
- [Part 2A — Pretrained Evaluation on CIFAR-10](#part2a)
- [Part 2B — From-Scratch Replication](#part2b)
  - `nn.Module` implementation
  - Parameter & output-shape verification
  - Weight init (Kaiming)
  - Training on CIFAR-10


<a id="part1"></a>
# Part 1 — Paper Review & Parameter Calculation

## 1.1 Architecture overview

ResNet-18 is an 18-layer convolutional network built from **basic residual blocks**. The core idea of the paper is the *identity shortcut*: each block computes $y = \mathcal{F}(x, \{W_i\}) + x$ instead of $y = \mathcal{F}(x, \{W_i\})$. This reformulation makes it easier to train very deep networks because the gradient can flow directly through the shortcut.

Following **Table 1 of the paper**, the ImageNet variant of ResNet-18 is structured as:

| Stage       | Output size   | Layers                                                      |
|-------------|---------------|-------------------------------------------------------------|
| conv1       | 112×112       | 7×7, 64, stride 2                                           |
| (maxpool)   | 56×56         | 3×3, stride 2                                               |
| conv2_x     | 56×56         | `[3×3, 64; 3×3, 64]` × 2                                    |
| conv3_x     | 28×28         | `[3×3, 128; 3×3, 128]` × 2 (first block stride 2)           |
| conv4_x     | 14×14         | `[3×3, 256; 3×3, 256]` × 2 (first block stride 2)           |
| conv5_x     | 7×7           | `[3×3, 512; 3×3, 512]` × 2 (first block stride 2)           |
| avgpool     | 1×1           | adaptive average pool                                       |
| fc          | 1000          | fully connected, 512 → 1000, softmax                        |

Counting: `conv1` (1) + 4 stages × 2 blocks × 2 convs each (16) + `fc` (1) = **18 weight layers**, which is where the name comes from. BN layers, activations, and the pooling layers are not counted toward the depth.

Whenever the channel count doubles or the spatial resolution halves, the identity shortcut cannot be added directly, so a **1×1 projection conv** (the "downsample" branch) is inserted on the skip connection to match dimensions. This happens at the start of `conv3_x`, `conv4_x`, and `conv5_x`.


## 1.2 Per-layer parameter formulas

I'll count **trainable parameters only**. What's in and what's out:

**Included**
- `Conv2d` weights: $C_{in} \times C_{out} \times k_h \times k_w$. In ResNet these convs have `bias=False` (BN absorbs bias).
- `BatchNorm2d` affine params: $\gamma$ and $\beta$, 2 per channel, i.e. $2C$.
- `Linear` weights + bias: $D_{in} \times D_{out} + D_{out}$.

**Excluded**
- BatchNorm's `running_mean` and `running_var` — these are buffers updated by EMA during training, not trained by gradient descent.
- `num_batches_tracked` — also a buffer.
- ReLU, MaxPool, AdaptiveAvgPool — parameter-free.
- Dropout masks (ResNet-18 has no dropout anyway).

### Formulas

$$
\text{Conv2d: } \quad P_{\text{conv}} = C_{in} \cdot C_{out} \cdot k_h \cdot k_w \; (+\, C_{out} \text{ if bias})
$$

$$
\text{BatchNorm2d: } \quad P_{\text{bn}} = 2 C
$$

$$
\text{Linear: } \quad P_{\text{fc}} = D_{in} \cdot D_{out} + D_{out}
$$

### BasicBlock building block

A `BasicBlock(in, out, stride)` has:

- conv1: 3×3, $C_{in} \to C_{out}$, stride = `stride` → $9 \cdot C_{in} C_{out}$ params
- bn1: $2 C_{out}$
- conv2: 3×3, $C_{out} \to C_{out}$, stride = 1 → $9 \cdot C_{out}^2$
- bn2: $2 C_{out}$
- downsample (only when $C_{in} \ne C_{out}$ or stride $\ne 1$): 1×1 conv, $C_{in} C_{out}$ + BN $2 C_{out}$


## 1.3 Layer-by-layer breakdown

The cell below computes the parameter count from these formulas alone. No PyTorch — just arithmetic. I'll cross-check against `torchvision.models.resnet18` in the next section.

In [ ]:
# ---- Manual parameter count from formulas (no PyTorch used here) ----
from dataclasses import dataclass
from typing import List, Tuple

def conv_params(cin, cout, k, bias=False):
    return cin * cout * k * k + (cout if bias else 0)

def bn_params(c):
    return 2 * c  # gamma + beta

def linear_params(din, dout, bias=True):
    return din * dout + (dout if bias else 0)

def basic_block_params(cin, cout, stride):
    """Returns (total_params, list_of_(name, params) rows)."""
    rows = []
    total = 0
    # conv1 3x3
    p = conv_params(cin, cout, 3);   total += p; rows.append((f"conv1 3x3 {cin}->{cout} s{stride}", p))
    p = bn_params(cout);             total += p; rows.append((f"bn1 ({cout})", p))
    # conv2 3x3
    p = conv_params(cout, cout, 3);  total += p; rows.append((f"conv2 3x3 {cout}->{cout}", p))
    p = bn_params(cout);             total += p; rows.append((f"bn2 ({cout})", p))
    # downsample projection, if any
    if stride != 1 or cin != cout:
        p = conv_params(cin, cout, 1); total += p; rows.append((f"  downsample conv 1x1 {cin}->{cout} s{stride}", p))
        p = bn_params(cout);           total += p; rows.append((f"  downsample bn ({cout})", p))
    return total, rows

def resnet18_manual():
    breakdown = []     # list of (stage, total, [subrows])
    grand_total = 0

    # ----- Stem -----
    stem_rows = []
    p = conv_params(3, 64, 7); stem_rows.append(("conv1 7x7 3->64 s2", p)); stem_total = p
    p = bn_params(64);         stem_rows.append(("bn1 (64)", p));           stem_total += p
    breakdown.append(("Stem", stem_total, stem_rows))
    grand_total += stem_total

    # ----- Stages -----
    stages = [
        ("conv2_x (layer1)", [(64, 64, 1), (64, 64, 1)]),
        ("conv3_x (layer2)", [(64, 128, 2), (128, 128, 1)]),
        ("conv4_x (layer3)", [(128, 256, 2), (256, 256, 1)]),
        ("conv5_x (layer4)", [(256, 512, 2), (512, 512, 1)]),
    ]
    for stage_name, blocks in stages:
        stage_total = 0
        stage_rows = []
        for i, (cin, cout, s) in enumerate(blocks, start=1):
            bt, brows = basic_block_params(cin, cout, s)
            stage_rows.append((f"  block{i}:", None))
            stage_rows.extend([("    " + n, p) for n, p in brows])
            stage_rows.append((f"  block{i} subtotal", bt))
            stage_total += bt
        breakdown.append((stage_name, stage_total, stage_rows))
        grand_total += stage_total

    # ----- Head -----
    head_rows = []
    p = linear_params(512, 1000, bias=True); head_rows.append(("fc 512->1000 (bias=True)", p))
    breakdown.append(("Head", p, head_rows))
    grand_total += p

    return grand_total, breakdown

manual_total, breakdown = resnet18_manual()

print(f"{'Layer':<50} {'Params':>14}")
print("-" * 66)
for stage, stotal, rows in breakdown:
    print(f"[{stage}]")
    for name, p in rows:
        if p is None:
            print(f"{name}")
        else:
            print(f"{name:<50} {p:>14,}")
    print(f"{'  → stage subtotal':<50} {stotal:>14,}")
    print()

print("=" * 66)
print(f"{'MANUAL TOTAL TRAINABLE PARAMETERS':<50} {manual_total:>14,}")


## 1.4 Summary table

A cleaner view, aggregated at the stage level:

| Stage | Output size | Formula (per block) | # blocks | Params |
|---|---|---|---|---|
| conv1 + bn1 | 112×112 | $7{\cdot}7{\cdot}3{\cdot}64 + 2{\cdot}64$ | — | 9,536 |
| conv2_x | 56×56 | $2(9{\cdot}64{\cdot}64) + 4{\cdot}64$ | 2 | 147,968 |
| conv3_x | 28×28 | first: $9{\cdot}64{\cdot}128 + 9{\cdot}128^2 + 4{\cdot}128 + 1{\cdot}64{\cdot}128 + 2{\cdot}128$; second: $2(9{\cdot}128^2) + 4{\cdot}128$ | 2 | 525,568 |
| conv4_x | 14×14 | analogous, channels 128→256 | 2 | 2,099,712 |
| conv5_x | 7×7 | analogous, channels 256→512 | 2 | 8,393,728 |
| fc | 1 | $512 \cdot 1000 + 1000$ | — | 513,000 |
| **Total** | | | | **11,689,512** |

Sanity check: the overwhelming majority of parameters live in `conv5_x` (512-channel 3×3 convs have $9 \cdot 512^2 \approx 2.36$M weights each), which matches intuition — later stages are where ResNet-18 spends its capacity.


## 1.5 Compare with torchvision's official implementation

Here I load torchvision's ResNet-18 (no weights needed — architecture alone) and compare.

In [ ]:
import torch
from torchvision.models import resnet18

model_tv = resnet18(weights=None)

tv_trainable = sum(p.numel() for p in model_tv.parameters() if p.requires_grad)
tv_all       = sum(p.numel() for p in model_tv.parameters())
tv_buffers   = sum(b.numel() for b in model_tv.buffers())

print(f"Manual count (formulas):                 {manual_total:>12,}")
print(f"Torchvision trainable parameters:        {tv_trainable:>12,}")
print(f"Torchvision total parameters (all):      {tv_all:>12,}")
print(f"Torchvision buffers (BN running stats):  {tv_buffers:>12,}")
print()
print(f"Exact match with manual count? {manual_total == tv_trainable}")


## 1.6 Discussion of discrepancies

The manual count should match torchvision's `parameters()` count **exactly**: 11,689,512.

If someone counts naively and gets a different number, these are the usual culprits:

1. **Counting conv biases.** ResNet's convs use `bias=False` because BN immediately follows and already has a learnable $\beta$ — two additive biases would be redundant. Adding biases inflates the count by ~5K.
2. **Counting BN running statistics as trainable.** `running_mean` and `running_var` are tracked as `buffers`, not `parameters`. They're updated by an exponential moving average during training, never by backprop. Torchvision reports ~9,600 buffer elements for ResNet-18 — those are not parameters.
3. **Forgetting the 1×1 downsample projections.** Three of them (at the start of `layer2`, `layer3`, `layer4`) contribute $64{\cdot}128 + 128{\cdot}256 + 256{\cdot}512 = 172{,}032$ params plus their BN. Easy to miss.
4. **FC bias.** The final `Linear(512, 1000)` *does* have a bias in torchvision, adding 1,000 params.

Reference values reported in the literature (e.g. torchvision's model docs, "Deep Residual Learning for Image Recognition") are typically quoted as **~11.7 M** for ResNet-18 — fully consistent with our exact 11,689,512.


<a id="part2a"></a>
# Part 2A — Pretrained Evaluation on CIFAR-10

## Strategy

ResNet-18's ImageNet weights are trained for 224×224 RGB images over 1000 classes. CIFAR-10 is 32×32 over 10 classes. To evaluate the *pretrained* weights meaningfully I:

1. **Upsample** CIFAR-10 test images to 224×224 with bilinear interpolation.
2. Normalize with **ImageNet mean/std** (that's what the pretrained model expects).
3. Run forward passes, get logits over 1000 ImageNet classes.
4. Map ImageNet classes to the 10 CIFAR-10 classes using a manually curated list of related WordNet IDs (e.g., `tabby cat`, `tiger cat`, `Egyptian cat` all count as "cat"). Predictions that don't fall into any CIFAR class are counted as wrong — this is a **zero-shot transfer** baseline, not a fine-tuned evaluation.

I'll report:
- **Top-1 and Top-5 raw ImageNet accuracy** — not meaningful for CIFAR labels directly, so I report it on a per-sample basis as "did the top-k predictions contain *any* class we mapped to the ground-truth CIFAR class?".
- **Zero-shot CIFAR-10 accuracy** using the mapped prediction.
- **Per-class classification report** for 3+ classes, plus confusion matrix.
- **Runtime & peak GPU memory.**


In [ ]:
import time, gc
import torch
import torch.nn as nn
import numpy as np
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(0)
np.random.seed(0)


In [ ]:
# ---- CIFAR-10 test set at 224x224 with ImageNet normalization ----
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize(224),           # bilinear by default
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=eval_transform
)
test_loader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck"]
print("CIFAR-10 test set size:", len(testset))


In [ ]:
# ---- Load pretrained ResNet-18 ----
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.IMAGENET1K_V1
pretrained = resnet18(weights=weights).to(device).eval()
print("Loaded pretrained ResNet-18 (IMAGENET1K_V1).")
print(f"Parameters: {sum(p.numel() for p in pretrained.parameters()):,}")


### Mapping ImageNet classes → CIFAR-10

I built a small map from CIFAR class → list of ImageNet class indices that semantically belong to it. This lets us do zero-shot evaluation: a prediction is "correct for CIFAR" if the top-1 ImageNet prediction is in the mapped set for the true CIFAR label.

The mapping is intentionally conservative — only clear matches. Anything not covered simply fails the classification, which is the honest outcome.

In [ ]:
# ImageNet class name -> index for the classes we care about.
# Indices are the standard 0-999 ImageNet ordering.
imagenet_to_cifar = {
    "airplane":   [404, 895],                                      # airliner, warplane
    "automobile": [436, 468, 511, 609, 627, 656, 661, 705, 717,
                   734, 751, 817, 829, 864, 867],                  # cars, taxis, limos, wagons, etc.
    "bird":       list(range(7, 25)) + [80, 81, 82, 83, 84, 85,
                   86, 87, 88, 89, 90, 91, 92, 93, 94, 95,
                   96, 97, 98, 99, 100],                            # various birds
    "cat":        [281, 282, 283, 284, 285],                       # tabby/tiger/Persian/Siamese/Egyptian cat
    "deer":       [351, 352, 353],                                 # hartebeest, impala, gazelle (closest)
    "dog":        list(range(151, 269)),                           # all dog breeds
    "frog":       [30, 31, 32],                                    # bullfrog, tree frog, tailed frog
    "horse":      [339, 340],                                      # sorrel, zebra (close)
    "ship":       [472, 554, 625, 628, 724, 814, 833, 914],        # canoe, fireboat, lifeboat, liner, etc.
    "truck":      [555, 569, 675, 717, 864, 867],                  # fire engine, garbage truck, pickup, etc.
}
# sanity
for k, v in imagenet_to_cifar.items():
    assert all(0 <= i < 1000 for i in v), k


In [ ]:
# ---- Inference loop with top-1 / top-5 in the *mapped* sense ----
cifar_to_imagenet = imagenet_to_cifar  # alias for clarity
n = len(testset)

correct_top1 = 0
correct_top5 = 0
y_true, y_pred = [], []  # for confusion matrix; predicted CIFAR label = argmax over mapped ImageNet sets

# GPU memory tracking
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

t0 = time.time()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        logits = pretrained(imgs)                       # (B, 1000)
        # Top-5 ImageNet indices per sample
        top5_idx = logits.topk(5, dim=1).indices.cpu().numpy()
        labels_np = labels.numpy()

        for i, lab in enumerate(labels_np):
            cname = CIFAR_CLASSES[lab]
            mapped = set(cifar_to_imagenet[cname])
            t1 = top5_idx[i, 0]
            top5 = set(top5_idx[i].tolist())

            if t1 in mapped:
                correct_top1 += 1
            if mapped & top5:
                correct_top5 += 1

            # For confusion matrix: assign each sample to whichever CIFAR class had
            # the largest mapped logit (a zero-shot prediction).
            # Efficient vectorized version would be cleaner; this is per-sample for readability.
            logits_i = logits[i].cpu().numpy()
            best_class, best_score = None, -1e18
            for c_name, c_idxs in cifar_to_imagenet.items():
                s = logits_i[c_idxs].max()
                if s > best_score:
                    best_score = s
                    best_class = c_name
            y_true.append(cname)
            y_pred.append(best_class)

elapsed = time.time() - t0
peak_mem_mb = (torch.cuda.max_memory_allocated(device) / 1e6) if device.type == "cuda" else None

print(f"Samples evaluated: {n}")
print(f"Top-1 (mapped):   {correct_top1/n:.4f}")
print(f"Top-5 (mapped):   {correct_top5/n:.4f}")
print(f"Runtime:          {elapsed:.2f} s  ({n/elapsed:.1f} img/s)")
if peak_mem_mb is not None:
    print(f"Peak GPU memory:  {peak_mem_mb:.1f} MB")


### Classification report + confusion matrix

I'll show the report for all 10 classes (well past the "≥3 classes" requirement) and a 3×3 confusion matrix zoom-in on `cat`, `dog`, `deer` — three classes that visually overlap and are interesting to compare.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("=== Zero-shot CIFAR-10 classification report ===")
print(classification_report(y_true, y_pred, labels=CIFAR_CLASSES, zero_division=0))

# 3-class zoom
zoom_classes = ["cat", "dog", "deer"]
yt = [y for y in y_true if y in zoom_classes]
yp = [p for y, p in zip(y_true, y_pred) if y in zoom_classes]
# Clip predictions to the zoom set (so off-set predictions still show up as "other")
cm = confusion_matrix(yt, yp, labels=zoom_classes)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(zoom_classes))); ax.set_xticklabels(zoom_classes)
ax.set_yticks(range(len(zoom_classes))); ax.set_yticklabels(zoom_classes)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix — cat / dog / deer")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()


**Expected result:** zero-shot accuracy lands around 40–60 % depending on how generous the class mapping is — nowhere near a fine-tuned model, but well above the 10 % random-guess baseline, which is exactly what you'd expect from an ImageNet-pretrained backbone with no fine-tuning. Dogs should score highest because ImageNet has 118 dog classes; deer will score worst because CIFAR "deer" has no dedicated ImageNet class (I mapped it to similar ungulates as an approximation).


<a id="part2b"></a>
# Part 2B — Replicate the Architecture from Scratch

## 2B.1 `nn.Module` implementation

No `torchvision.models`. Just `nn.Conv2d`, `nn.BatchNorm2d`, `nn.ReLU`, `nn.Linear`, `nn.MaxPool2d`, `nn.AdaptiveAvgPool2d` — the building blocks the assignment allows.

I'll match torchvision's naming conventions (`conv1`, `bn1`, `layer1…layer4`, `fc`) so the parameter count and `state_dict` structure are directly comparable.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    """The basic residual block used by ResNet-18 / ResNet-34.

    Structure (He et al., 2015):
        x -> conv 3x3 -> BN -> ReLU -> conv 3x3 -> BN
        residual = x  (or 1x1 conv projection if shape mismatch)
        out = ReLU(F(x) + residual)
    """
    expansion = 1   # kept for API parity with Bottleneck, not used here

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)

        # Identity shortcut OR 1x1 projection when dims don't line up
        if stride != 1 or in_planes != planes * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )
        else:
            self.downsample = None

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return F.relu(out, inplace=True)


class ResNet18Scratch(nn.Module):
    """From-scratch replica of ImageNet ResNet-18 (He et al., 2015, Table 1)."""

    def __init__(self, num_classes: int = 1000):
        super().__init__()
        self.in_planes = 64

        # ---- Stem ----
        self.conv1   = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ---- 4 stages, 2 BasicBlocks each ----
        self.layer1 = self._make_layer(64,  blocks=2, stride=1)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=2)
        self.layer4 = self._make_layer(512, blocks=2, stride=2)

        # ---- Head ----
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(512 * BasicBlock.expansion, num_classes)

        # Kaiming init (He et al. initialization), as in the paper
        self._init_weights()

    def _make_layer(self, planes: int, blocks: int, stride: int) -> nn.Sequential:
        layers = [BasicBlock(self.in_planes, planes, stride)]
        self.in_planes = planes * BasicBlock.expansion
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_planes, planes, stride=1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


## 2B.2 Verify parameter count and output shapes

This is where Part 1 and Part 2 meet: the manual count, the torchvision count, and my scratch implementation should all agree to the integer.

In [ ]:
scratch = ResNet18Scratch(num_classes=1000)
scratch_params = sum(p.numel() for p in scratch.parameters() if p.requires_grad)

print(f"Manual (Part 1):       {manual_total:>12,}")
print(f"Torchvision:           {tv_trainable:>12,}")
print(f"Scratch replica:       {scratch_params:>12,}")
print(f"All three match? {manual_total == tv_trainable == scratch_params}")

# Forward-pass shape checks
scratch.eval()
with torch.no_grad():
    y224 = scratch(torch.randn(1, 3, 224, 224))
    y32  = scratch(torch.randn(1, 3, 32,  32))
print(f"\nOutput shape for (1, 3, 224, 224): {tuple(y224.shape)}")
print(f"Output shape for (1, 3,  32,  32): {tuple(y32.shape)}")


A note on the 32×32 input: the stem has stride 2 + maxpool stride 2, so spatial size becomes 8×8 after the stem, then 4×4, 2×2, 1×1, 1×1. It technically works but most of the capacity of the network is wasted — for real CIFAR-10 training people use the **CIFAR variant** of ResNet (3×3 stem, no maxpool). For this assignment I replicate the **ImageNet variant** faithfully and adapt to CIFAR by upsampling.

## 2B.3 Optional sanity check — load pretrained weights into the scratch model

If my layer naming mirrors torchvision's, I should be able to load the pretrained `state_dict` directly. This is a powerful correctness check — any mismatch would show up instantly.

In [ ]:
scratch_for_load = ResNet18Scratch(num_classes=1000)
pretrained_sd = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).state_dict()

missing, unexpected = scratch_for_load.load_state_dict(pretrained_sd, strict=False)
print("Missing keys (in scratch, not in pretrained):", missing)
print("Unexpected keys (in pretrained, not in scratch):", unexpected)

# If both lists are empty, the naming matches torchvision exactly.
# With strict=True it would just work:
scratch_for_load.load_state_dict(pretrained_sd, strict=True)
print("\nstrict=True load succeeded — naming is 100% compatible.")


## 2B.4 Train the scratch model on CIFAR-10

I'll train from scratch (no pretrained init) for a small number of epochs — enough to show the model learns and to report honest CIFAR-10 accuracy. On a single modern GPU, ~10 epochs at 224×224 take several minutes; I'll use 32×32 input to keep training fast, with a slightly modified num_classes=10 head.

In [ ]:
# CIFAR-10 at 32x32 with standard augmentation
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])
test_transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

trainset_c = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_transform)
testset_c  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_transform_cifar)
train_loader = DataLoader(trainset_c, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
cifar_test_loader = DataLoader(testset_c, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Fresh scratch model with 10 classes
model = ResNet18Scratch(num_classes=10).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()

print(f"Training ResNet18Scratch (num_classes=10) on CIFAR-10 32x32")
print(f"Trainable params: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
EPOCHS = 10

def evaluate(m, loader):
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = m(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    n_seen = 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * y.size(0)
        n_seen += y.size(0)
    scheduler.step()
    train_loss = running_loss / n_seen
    val_acc = evaluate(model, cifar_test_loader)
    history.append((epoch, train_loss, val_acc))
    print(f"Epoch {epoch:2d}/{EPOCHS} | loss {train_loss:.4f} | val acc {val_acc:.4f} | {time.time()-t0:.1f}s")

final_acc = evaluate(model, cifar_test_loader)
print(f"\nFinal CIFAR-10 test accuracy: {final_acc:.4f}")


In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
ep, tl, va = zip(*history)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(ep, tl, "o-", label="train loss", color="tab:red")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(ep, va, "s-", label="val acc", color="tab:blue")
ax2.set_ylabel("val accuracy", color="tab:blue")
plt.title("ResNet18Scratch on CIFAR-10 (32x32)")
fig.tight_layout(); plt.show()


In [ ]:
# Final per-class report on CIFAR-10
model.eval()
y_true_c, y_pred_c = [], []
with torch.no_grad():
    for x, y in cifar_test_loader:
        x = x.to(device)
        y_pred_c.extend(model(x).argmax(1).cpu().tolist())
        y_true_c.extend(y.tolist())

y_true_names = [CIFAR_CLASSES[i] for i in y_true_c]
y_pred_names = [CIFAR_CLASSES[i] for i in y_pred_c]
print(classification_report(y_true_names, y_pred_names, labels=CIFAR_CLASSES, digits=3))


# Summary

| Deliverable | Result |
|---|---|
| **Part 1** — manual param count | **11,689,512** |
| **Part 1** — matches torchvision `resnet18` | ✅ (exact) |
| **Part 2A** — pretrained zero-shot CIFAR-10 (mapped) | see report above |
| **Part 2B** — scratch param count | **11,689,512** (same integer) |
| **Part 2B** — scratch loads pretrained `state_dict` with `strict=True` | ✅ |
| **Part 2B** — CIFAR-10 test accuracy after 10 epochs | see final report |

## Key takeaways

1. The ResNet-18 parameter count falls out exactly from three simple formulas applied carefully — no hidden magic, just attention to `bias=False` on the convs and to the BN-vs-buffer distinction.
2. Most of the capacity (~70 %) lives in `layer4`; the 512-channel 3×3 convs dominate.
3. Naming layers the same way torchvision does pays off: the scratch replica is a true drop-in — you can hot-swap the pretrained `state_dict` into it with `strict=True`, which is the strongest possible correctness check.
4. The ImageNet ResNet-18 architecture trained from random init on 32×32 CIFAR-10 reaches reasonable accuracy but is not optimal for that resolution — the CIFAR-specific variant (3×3 stem, no maxpool) is what the paper actually recommends for small images.

## References

- He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR. arXiv:1512.03385.
- torchvision documentation: `torchvision.models.resnet18`, `torchvision.models.ResNet18_Weights`.
